![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Arabic Speech-to-Text & Subtitle Generation

in this lab we'll explore STT in two tasks.
1. transcript generation of a video clip.
2. subtitle generation in Arabic of a tv series clip.

**Audio data:** I've uploaded two clips ! first one is a part of a Math course I have on infinte series, specificly the Ratio Test.
second one is a random clip from my favorite Sitcom Cheers. we'll create a transcript of the first clip and an SRT (subtitle file) of the second one.

---

### Before you run anything

1. **Accept the gated model's terms.** Open [CohereLabs/cohere-transcribe-arabic-07-2026](https://huggingface.co/CohereLabs/cohere-transcribe-arabic-07-2026) on hugging face, log in, and accept the conditions. Without this, Part 1 fails with a 403.
2. **Enable a GPU.** Runtime → Change runtime type → GPU. An A100 or L4 is comfortable.


In [1]:
%pip install -q \
    transformers==5.15.0 \
    accelerate==1.14.0 \
    huggingface_hub==1.27.0 \
    sentencepiece==0.2.2 \
    protobuf==6.33.6 \
    faster-whisper==1.2.1 \
    ctranslate2==4.8.1 \
    soundfile==0.13.1 \
    librosa==0.11.0 \
    srt==3.5.3

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 53.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.


### 0.1 — Verify the environment
Sometimes when installing libraries, they can downgrade each other or ruin compatibility or mess things up (dependancy hell).


If versions are going to fight, this cell fails in two seconds instead of thirty frames deep inside a model load. (it could still fail, but it catches the easy stuff)

In [2]:
import numpy, torch, transformers, librosa, soundfile, srt, ctranslate2
from faster_whisper import WhisperModel

print(f"numpy        {numpy.__version__}")
print(f"torch        {torch.__version__}")
print(f"transformers {transformers.__version__}")
print(f"librosa      {librosa.__version__}")
print(f"ctranslate2  {ctranslate2.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    raise RuntimeError("No GPU detected. Runtime -> Change runtime type -> GPU.")
print("\nEnvironment OK")

numpy        2.0.2
torch        2.11.0+cu128
transformers 5.15.0
librosa      0.11.0
ctranslate2  4.8.1
CUDA available: True
GPU: Tesla T4 (15.6 GB)

Environment OK


### 0.2 — Hugging Face authentication

The Cohere Arabic ASR model is gated, so a token is required. This is a **Hub token, not an inference API key** — nothing in this notebook calls a hosted service. Everything runs on this machine. Everything is local.

In [3]:
from huggingface_hub import login, whoami

login()  # paste a token with 'read' scope

print(f"Logged in as: {whoami()['name']}")

Logged in as: frost000


### 0.3 — Configuration

Every tunable lives here. Nothing below this cell hardcodes a value.

In [4]:
import os
from pathlib import Path

# --- models
ASR_ARABIC_MODEL = "CohereLabs/cohere-transcribe-arabic-07-2026"
WHISPER_MODEL    = "large-v3"
TRANSLATION_MODEL = "Qwen/Qwen3-4B-Instruct-2507"

# --- data
REPO = "frost000/KAUST_STT_Lab"
RATIO_FILE  = "Ratio_Test_clip.mp3"
CHEERS_FILE = "cheers_clip.mp3"

# --- audio
TARGET_SR = 16_000          # both models expect 16 kHz mono

# --- transcription
ARABIC_LANGUAGE = "ar"      # language tag for the Cohere processor
SOURCE_LANGUAGE = "en"      # Cheers clip is English

# --- translation
BATCH_SIZE   = 20           # subtitle cues sent to the LLM per call
CONTEXT_OVERLAP = 2         # cues carried over from the previous batch
MAX_CHARS_PER_LINE = 42     # Arabic reads slower than the equivalent English line

# --- output
OUTPUT_DIR = Path("/content/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("HF_HOME", "/content/hf_cache")

print(f"Outputs -> {OUTPUT_DIR}")

Outputs -> /content/outputs


### 0.4 — Small helpers

A timer so each stage reports how long it took, and a VRAM release helper. to run the notebook on a T4 gpu we need to free the VRAM after each model usage ...

In [5]:
import gc, time
from contextlib import contextmanager

import torch

@contextmanager
def stage(name: str):
    '''Time a stage and print how long it took.'''
    print(f"[{name}] started")
    t0 = time.time()
    yield
    print(f"[{name}] finished in {time.time() - t0:.1f}s\n")


def free_vram(*objects) -> None:
    '''Delete model objects and return their memory to the GPU.'''
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()
    used = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM in use after cleanup: {used:.2f} GB")

## 1 — Load the audio

Two files, pulled straight from the Hub. `hf_hub_download` returns a local path, and `librosa.load` decodes to a mono float32 array at 16 kHz.

That gives us both shapes we need downstream: **a file path** for `faster-whisper`, and **a numpy array** for the Cohere processor.

In [6]:
from huggingface_hub import hf_hub_download
import librosa

ratio_path  = hf_hub_download(REPO, RATIO_FILE,  repo_type="dataset")
cheers_path = hf_hub_download(REPO, CHEERS_FILE, repo_type="dataset")

ratio_audio, sr = librosa.load(ratio_path, sr=TARGET_SR, mono=True)
print(ratio_audio.shape, sr, ratio_audio.dtype)

Ratio_Test_clip.mp3: reconstructing file:   0%|          |  0.00B / 1.44MB            

Ratio_Test_clip.mp3: downloading bytes:           |  0.00B            

cheers_clip.mp3: reconstructing file:   0%|          |  0.00B / 1.24MB            

cheers_clip.mp3: downloading bytes:           |  0.00B            

(2880000,) 16000 float32


In [7]:
from IPython.display import Audio, display

for label, path in [("Ratio test (Arabic)", ratio_path), ("Cheers (English)", cheers_path)]:
    duration = librosa.get_duration(path=path)
    print(f"{label}: {duration:.1f}s")
    display(Audio(path))

Ratio test (Arabic): 180.0s


Cheers (English): 155.0s


## 2 — Part 1: Arabic ASR with a specialised model

`cohere-transcribe-arabic-07-2026` is a 2B Conformer encoder–decoder, Apache 2.0, fine-tuned from Cohere's multilingual Transcribe model specifically for Arabic

It is supported natively in `transformers` as `CohereAsrForConditionalGeneration`.

**Three limitations worth knowing before you build on it:**

- **No timestamps.** You get a text blob. no mention of when this phrase was said.
- **No speaker diarization.** Two people talking produce one undifferentiated transcript.
- **It transcribes eagerly.** Non-speech audio tends to come back as text rather than silence, so production pipelines put a VAD or noise gate upstream.


In [8]:
#helper function to print the output transcription
import textwrap, re
def show_transcript(text: str, width: int = 90):
    """Print a transcript wrapped into readable lines, one sentence per block."""
    for sentence in re.split(r"(?<=[.!?؟।])\s+", text.strip()):
        if sentence.strip():
            print(textwrap.fill(sentence.strip(), width=width))
            print()

In [9]:
from transformers import AutoProcessor, CohereAsrForConditionalGeneration

with stage("load Cohere Arabic ASR"):
    processor = AutoProcessor.from_pretrained(ASR_ARABIC_MODEL)
    asr_model = CohereAsrForConditionalGeneration.from_pretrained(
        ASR_ARABIC_MODEL,
        device_map="auto",
        dtype=torch.bfloat16,
    )

print(f"Parameters: {asr_model.num_parameters() / 1e9:.2f}B")

[load Cohere Arabic ASR] started


processor_config.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.13GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2150 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

[load Cohere Arabic ASR] finished in 57.3s

Parameters: 2.07B


In [10]:
#truncate ratio clip to ~35s starting from clip_s
clip_s = 35
ratio_audio = ratio_audio[clip_s*TARGET_SR : (clip_s+35) * TARGET_SR]

In [11]:
with stage("transcribe ratio test clip"):
    inputs = processor(
        ratio_audio,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
        language=ARABIC_LANGUAGE
    )
    inputs = inputs.to(asr_model.device, dtype=asr_model.dtype)

    generated = asr_model.generate(**inputs, max_new_tokens=512)
    cohere_transcript = processor.decode(generated[0], skip_special_tokens=True)

show_transcript(cohere_transcript)

[transcribe ratio test clip] started
[transcribe ratio test clip] finished in 4.8s

فوق عندنا كيتك عيب تحت عندنا هذا في هذا فعندنا كيتك عيب فهذا بالنهايه بيطلع لنا ان الليميت
لما تحلونه اذا ودكم يعني اذا مره يعني حلوه بيطلع معاكم كيتك عيب على كيتك عيب خلاصة انه
بيطلع معاكم واحد واحد يعني نو كونكلوجن الدوال الكسريه اللي زي كذا بشكل عام حاولوا انكم ما
تحرصون عليها بالريشو تيست لان الريشو تيست افضل لكم بالاشياء الاسية فاكتوريز اشياء معقدة
انها بتكنسل نفسها بنفسها فاكتوريز بالعاده قصدي البولينوميز اللي زي كذا



In [12]:
(OUTPUT_DIR / "ratio_test_cohere_ar.txt").write_text(cohere_transcript, encoding="utf-8")
free_vram(asr_model, processor)

VRAM in use after cleanup: 4.14 GB


### 2.1 — Baseline: the same clip through a general multilingual model

The comparison is the point of this section. Whisper large-v3 is a strong general model covering ~100 languages; the Cohere model is 2B parameters aimed at one. Run both on identical audio and judge for yourself.

**What to look for:**

- **Code-switching.** The clip mixes Arabic speech with English mathematical terms. Does each model transliterate the English into Arabic script, keep it in Latin script, or mangle it?
- **Technical vocabulary.** Is "ratio test" recognised as a term, or transcribed phonetically?
- **Orthography.** Hamza placement, taa marbuta vs haa, and whether diacritics appear at all.

In [13]:
from faster_whisper import WhisperModel

with stage("load faster-whisper large-v3"):
    whisper = WhisperModel(WHISPER_MODEL, device="cuda", compute_type="float16")

with stage("transcribe ratio test clip (Whisper)"):
    segments, info = whisper.transcribe(ratio_audio, language=ARABIC_LANGUAGE, beam_size=5)
    whisper_arabic = " ".join(seg.text.strip() for seg in segments)

print(f"Detected language: {info.language} (p={info.language_probability:.2f})\n")
show_transcript(whisper_arabic)

[load faster-whisper large-v3] started
[load faster-whisper large-v3] finished in 72.8s

[transcribe ratio test clip (Whisper)] started
[transcribe ratio test clip (Whisper)] finished in 5.8s

Detected language: ar (p=1.00)

فوق عندنا كي تكعيب تحت عندنا هذا فهذا فعندنا كي تكعيب فهذا بالنهاية بيطلع لنا أن الليميت
لما تحلونا إذا مرة يعني حلو بيطلع معاكم كي تكعيب على كي تكعيب خلاصة أنه بيطلع معاكم واحد
واحد يعني no conclusion الدوهان الكسرية اللي زي كذا بشكل عام حاولوا أنكم ما تحرصون عليها
بالريشو تست لأن الريشو تست أفضل لكم بالأشياء الأسية فاكتوريا الأشياء معقدة أنها بتكنسل
نفسها بنفسها فاكتوريالز بالعادة أقصدي البولينوميالز وزي كذا



In [14]:
(OUTPUT_DIR / "ratio_test_whisper_ar.txt").write_text(whisper_arabic, encoding="utf-8")

413

In [15]:
print("="*50)
print("Cohere STT:")
print("="*50)
show_transcript(cohere_transcript)

print("="*50)
print("Whisper STT:")
print("="*50)
show_transcript(whisper_arabic)

print(f"TTS of second {clip_s}s to {clip_s+35}s")
display(Audio(ratio_audio, rate=TARGET_SR))

Cohere STT:
فوق عندنا كيتك عيب تحت عندنا هذا في هذا فعندنا كيتك عيب فهذا بالنهايه بيطلع لنا ان الليميت
لما تحلونه اذا ودكم يعني اذا مره يعني حلوه بيطلع معاكم كيتك عيب على كيتك عيب خلاصة انه
بيطلع معاكم واحد واحد يعني نو كونكلوجن الدوال الكسريه اللي زي كذا بشكل عام حاولوا انكم ما
تحرصون عليها بالريشو تيست لان الريشو تيست افضل لكم بالاشياء الاسية فاكتوريز اشياء معقدة
انها بتكنسل نفسها بنفسها فاكتوريز بالعاده قصدي البولينوميز اللي زي كذا

Whisper STT:
فوق عندنا كي تكعيب تحت عندنا هذا فهذا فعندنا كي تكعيب فهذا بالنهاية بيطلع لنا أن الليميت
لما تحلونا إذا مرة يعني حلو بيطلع معاكم كي تكعيب على كي تكعيب خلاصة أنه بيطلع معاكم واحد
واحد يعني no conclusion الدوهان الكسرية اللي زي كذا بشكل عام حاولوا أنكم ما تحرصون عليها
بالريشو تست لأن الريشو تست أفضل لكم بالأشياء الأسية فاكتوريا الأشياء معقدة أنها بتكنسل
نفسها بنفسها فاكتوريالز بالعادة أقصدي البولينوميالز وزي كذا

TTS of second 35s to 70s


## 3 — Part 2: English → Arabic subtitles

Now the subtitle pipeline, on the Cheers clip. Three stages:

1. **Transcribe with timestamps**: use whisper large-v3, to generate timestemps + transcript.
2. **Translate with context**: use Qwen3-4B-Instruct, to translate sentances, batched, never cue-by-cue.
3. **Write a correct Arabic SRT File**: make sure reading speed is pleasing, format is correct as a standard SRT file.


### 3.1 — Transcription with word-level timestamps

`word_timestamps=True` matters for subtitles. Whisper's default segment boundaries often land mid-sentence; word timings let us trim each cue to the actual first and last spoken word, so cues start when someone starts talking.

In [16]:
with stage("transcribe Cheers clip"):
    segments, info = whisper.transcribe(
        cheers_path,
        language=SOURCE_LANGUAGE,
        beam_size=5,
        word_timestamps=True,
    )
    segments = list(segments)

print(f"{len(segments)} segments, detected language: {info.language}\n")
for seg in segments[:5]:
    print(f"[{seg.start:6.2f} -> {seg.end:6.2f}]  {seg.text.strip()}")

[transcribe Cheers clip] started
[transcribe Cheers clip] finished in 43.8s

74 segments, detected language: en

[  1.24 ->   3.68]  Cheers is filmed before a live studio audience.
[  4.02 ->   6.22]  Hey, Libby. Hey, Don. How's your day?
[  6.46 ->   8.56]  Ah, boring. Nothing happens in this brig anymore.
[  8.82 ->  10.18]  I know what you mean. I am bored.
[ 10.60 ->  12.10]  Oh, look at him. Look what we've got here.


Build SRT cues. Where word timings exist we tighten the cue to the first and last word, which removes the leading and trailing silence Whisper tends to include.

In [17]:
from datetime import timedelta
from typing import List

import srt
def segments_to_cues(segments) -> List[srt.Subtitle]:
    '''Convert Whisper segments into SRT cues, tightened to word boundaries.'''
    cues = []
    for seg in segments:
        text = seg.text.strip()
        if not text:
            continue
        start, end = seg.start, seg.end
        if seg.words:
            start = seg.words[0].start
            end = seg.words[-1].end
        # index is assigned after filtering, so dropped empty segments
        # never leave gaps that would desync the translation mapping
        cues.append(
            srt.Subtitle(
                index=len(cues) + 1,
                start=timedelta(seconds=start),
                end=timedelta(seconds=end),
                content=text,
            )
        )
    return cues


cues_en = segments_to_cues(segments)
srt_en = srt.compose(cues_en)

(OUTPUT_DIR / "cheers_en.srt").write_text(srt_en, encoding="utf-8")
print(f"{len(cues_en)} cues written\n")
print(srt_en[:600])

74 cues written

1
00:00:01,240 --> 00:00:03,680
Cheers is filmed before a live studio audience.

2
00:00:04,020 --> 00:00:06,220
Hey, Libby. Hey, Don. How's your day?

3
00:00:06,460 --> 00:00:08,560
Ah, boring. Nothing happens in this brig anymore.

4
00:00:08,820 --> 00:00:10,180
I know what you mean. I am bored.

5
00:00:10,600 --> 00:00:12,100
Oh, look at him. Look what we've got here.

6
00:00:12,220 --> 00:00:12,800
What? Look.

7
00:00:14,800 --> 00:00:16,380
Oh, it's the film critic, Channel 11.

8
00:00:16,600 --> 00:00:18,100
Oh, it's the anchorman, Channel 8.

9
00:00:19,880 --> 00:00:21,060
Do you


In [18]:
free_vram(whisper)

VRAM in use after cleanup: 4.14 GB


### 3.2 — Context-aware translation

The obvious implementation translates one cue (sentance) at a time. It produces bad Arabic, and the reason is structural rather than a matter of model quality.

A subtitle cue is a *fragment*. `"I told her about it"` carries no information about who is speaking, gender or many other attributes that matter in Arabic. English lets you stay vague. Arabic does not, the verb must agree in gender and number, and the pronoun has to be chosen. Given a fragment, the model has to guess, and it guesses masculine singular by default.

So we batch. Each request contains ~20 consecutive cues plus a couple carried from the previous batch, as a JSON array of `{id, text}`. The model sees the conversation and returns the same ids in the same order. **Timings are never touched**, we swap text onto existing cues.

Qwen3-4B-Instruct-2507 is a good fit here: it's small enough (no quanitization) to fit on a T4. ungated, and a non-thinking model, so it emits no `<think>` blocks to strip before parsing JSON.

In [19]:
from transformers import AutoModelForCausalLM, AutoTokenizer

with stage("load Qwen3-4B-Instruct"):
    tokenizer = AutoTokenizer.from_pretrained(TRANSLATION_MODEL)
    llm = AutoModelForCausalLM.from_pretrained(
        TRANSLATION_MODEL,
        dtype=torch.bfloat16,
        device_map="auto",
    )

print(f"Parameters: {llm.num_parameters() / 1e9:.2f}B")

[load Qwen3-4B-Instruct] started


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

[load Qwen3-4B-Instruct] finished in 382.4s

Parameters: 4.02B


In [20]:
import json
import re
from typing import Dict

SYSTEM_PROMPT = (
    "You are a professional subtitle translator. You translate English sitcom "
    "dialogue into natural spoken Arabic — the register Arabic-language streaming "
    "services use for comedy, not formal written Arabic."
)

INSTRUCTIONS = '''translate into natural conversational Arabic that's idiomatic;
jokes should land as jokes, not as literal translations.
Do not create literal translations, translate the gist of the meaning.

These lines are consecutive dialogue from a single scene. Use the surrounding lines to
resolve pronoun gender, number, and level of formality — a line in isolation is ambiguous,
the scene is not.

Rules:
- Keep each translation short enough to read on screen (aim for under {max_chars} characters).
- Translate the meaning, not word by word. Idioms become natural Arabic idioms.
- Keep proper nouns recognisable.
- Do not merge or split lines.

Return ONLY a JSON array of objects with keys "id" and "text", with the same ids in the
same order. No markdown fences, no commentary.

Input:
{payload}'''


def extract_json_array(raw: str) -> list:
    '''Pull the first JSON array out of a model response.'''
    match = re.search(r"\[.*\]", raw, flags=re.DOTALL)
    if match is None:
        raise ValueError(f"No JSON array found in model output:\n{raw[:400]}")
    return json.loads(match.group(0))


def translate_batch(batch: List[srt.Subtitle]) -> Dict[int, str]:
    '''Translate one batch of cues, returning {cue_index: arabic_text}.'''
    payload = [{"id": cue.index, "text": cue.content.replace("\n", " ")} for cue in batch]
    prompt = INSTRUCTIONS.format(
        max_chars=MAX_CHARS_PER_LINE,
        payload=json.dumps(payload, ensure_ascii=False, indent=2),
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(llm.device)

    generated = llm.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    reply = tokenizer.decode(
        generated[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    return {item["id"]: item["text"] for item in extract_json_array(reply)}

In [21]:
with stage("translate subtitles"):
    translations: Dict[int, str] = {}
    for start_idx in range(0, len(cues_en), BATCH_SIZE):
        context_start = max(0, start_idx - CONTEXT_OVERLAP)
        batch = cues_en[context_start:start_idx + BATCH_SIZE]
        translations.update(translate_batch(batch))
        print(f"  cues {batch[0].index}-{batch[-1].index} done "
              f"({len(translations)}/{len(cues_en)})")

missing = [cue.index for cue in cues_en if cue.index not in translations]
if missing:
    print(f"\nWARNING: {len(missing)} cues missing a translation: {missing}")
else:
    print("\nAll cues translated.")

[translate subtitles] started
  cues 1-20 done (20/74)
  cues 19-40 done (40/74)
  cues 39-60 done (60/74)
  cues 59-74 done (74/74)
[translate subtitles] finished in 949.4s


All cues translated.


In [22]:
free_vram(llm)

VRAM in use after cleanup: 10.98 GB


### 3.3 — Writing a correct Arabic SRT

Three details that decide whether the file renders properly, none of which are about translation quality:

- **UTF-8 with BOM.** Some players still guess the encoding wrong on Arabic text without it.
- **RLM marks.** A line mixing Arabic with Latin script or digits can render with its segments reordered. A right-to-left mark (`\u200f`) at the start of the line pins the base direction.
- **Reading speed.** Arabic script is denser than Latin, so an English line's worth of text takes longer to read. We flag any cue over the configured limit rather than silently shipping unreadable subtitles.

In [23]:
import copy

RLM = "\u200f"
LATIN_OR_DIGIT = re.compile(r"[A-Za-z0-9]")


def finalise_arabic_cue(text: str) -> str:
    '''Add a right-to-left mark when a line mixes scripts.'''
    text = text.strip()
    if LATIN_OR_DIGIT.search(text):
        text = RLM + text
    return text


cues_ar = copy.deepcopy(cues_en)
for cue in cues_ar:
    cue.content = finalise_arabic_cue(translations.get(cue.index, cue.content))

srt_ar = srt.compose(cues_ar)
(OUTPUT_DIR / "cheers_ar.srt").write_text(srt_ar, encoding="utf-8-sig")  # BOM

too_long = [c for c in cues_ar if len(c.content.lstrip(RLM)) > MAX_CHARS_PER_LINE]
print(f"{len(cues_ar)} cues written to cheers_ar.srt")
print(f"{len(too_long)} cues exceed {MAX_CHARS_PER_LINE} characters "
      f"and may be hard to read at speed")
for cue in too_long[:5]:
    print(f"  cue {cue.index}: {len(cue.content)} chars")

74 cues written to cheers_ar.srt
0 cues exceed 42 characters and may be hard to read at speed


### 3.4 — Review

Anyone who reads Arabic will spot gender-agreement errors, it's one of the drawbacks of having a ~bad machine translation model + no visual input, just a transcript, which is needed to translate (especially in sitcoms)

**Also worth noting: errors propagate.** A proper noun Whisper mishears in English becomes a confidently mistranslated one in Arabic. Showing both columns is how you tell a transcription fault from a translation fault.

In [24]:
import pandas as pd
#helper function for start
def fmt(td) -> str:
    total = td.total_seconds()
    return f"{int(total // 60):02d}:{total % 60:05.2f}"


review = pd.DataFrame({
    "#": [c.index for c in cues_en],
    "start": [fmt(c.start) for c in cues_en],
    "english": [c.content for c in cues_en],
    "arabic": [c.content.lstrip(RLM) for c in cues_ar],
})

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 200)
display(review)

,#,start,english,arabic
0,1,00:01.24,Cheers is filmed before a live studio audience.,Cheers يُُسجّل أمام جمهور حي.
1,2,00:04.02,"Hey, Libby. Hey, Don. How's your day?",مرحباً ليببي، مرحباً دون. كيف تسير يومك؟
2,3,00:06.46,"Ah, boring. Nothing happens in this brig anymore.",أوه، ممل. لا شيء يحصل في هذا المبنى!
3,4,00:08.82,I know what you mean. I am bored.,أعرف ما تقصده. أنا متعب.
4,5,00:10.60,"Oh, look at him. Look what we've got here.",أوه، انتظروا! ماذا نرى هنا؟
5,6,00:12.22,What? Look.,ما؟ انتظروا!
6,7,00:14.80,"Oh, it's the film critic, Channel 11.",أوه، هو المُراجع المُذيع من القناة 11.
7,8,00:16.60,"Oh, it's the anchorman, Channel 8.",أوه، هو المُذيع من القناة 8.
8,9,00:19.88,"Do you want to, uh...",هل ترغب أن تُقدّم...
9,10,00:21.06,"Here, your autograph?",هنا، توقيعك؟


## 4 — Outputs

In [25]:
print("Files written:\n")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {path.name:28} {path.stat().st_size:>8,} bytes")

print("\nDownload from the file browser in the left sidebar, or:")
print("  from google.colab import files; files.download('/content/outputs/cheers_ar.srt')")

Files written:

  cheers_ar.srt                   4,884 bytes
  cheers_en.srt                   4,283 bytes
  ratio_test_cohere_ar.txt          776 bytes
  ratio_test_whisper_ar.txt         742 bytes

Download from the file browser in the left sidebar, or:
  from google.colab import files; files.download('/content/outputs/cheers_ar.srt')


## 5 — Exercises

1. **Batch size ablation.** Set `BATCH_SIZE = 1` and rerun the translation stage, then diff it against the batched output. Count the gender-agreement errors in each. This isolates the effect of context from the effect of model size.

2. **Change the target register.** Modify `INSTRUCTIONS` to request Gulf dialect rather than Modern Standard Arabic. Which lines improve? Sitcom dialogue in MSA reads stiff — where does the register mismatch hurt most?

3. **Give the model the scene.** Add a one-line scene description to `SYSTEM_PROMPT` (who is speaking, their relationship). Does it fix pronoun errors the batching alone could not?

4. **Cross-model check.** Run the Cheers clip through the Cohere Arabic model. It expects Arabic; feeding it English tells you something about how ASR models fail when their language assumption is violated.
---
# Lab Cooked by yours truly